In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime, timedelta
# import datetime
import pandas as pd
import time
from tqdm import tqdm

options = Options()
options.add_experimental_option("detach", True)
options.add_argument("start-maximized")
options.add_argument("Chrome/135.0.0.0")
options.add_argument("lang=ko_KR")

# 웹브라우저가 백그라운드에서 작동하도록 설정
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
    )

url = "https://play.google.com/store/apps/details?id=viva.republica.toss"
driver.get(url)

# 평점 및 리뷰 옆의 -> 버튼 클릭
wait = WebDriverWait(driver, 10)
button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label*='평점 및 리뷰 자세히 알아보기']")))
button.click()

# 최신순으로 리뷰를 정렬하기 위해서 버튼 클릭
wait = WebDriverWait(driver, 10)
button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
button.click()

time.sleep(1)

# 최신을 찾아 클릭
wait = WebDriverWait(driver, 10)
button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "span[aria-label*='최신']")))
button.click()

time.sleep(3)

# 오늘로부터 3년전 날짜 만들기
today = datetime.today()
end_date = today - timedelta(days=10)
current_date = ""
reviews = ""

while True:
    
    try:
        # 리뷰가 담긴 창을 찾아서 JavaScript로 1000px씩 아래로 스크롤
        driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 1000)")
        time.sleep(1)

        # 리뷰들 중 가장 마지막 데이터의 날짜 추출해 날짜 데이터로 변환
        reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
        current_date = reviews[-1].find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
        current_date = current_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        current_date = datetime.strptime(current_date, "%Y-%m-%d")
        print(f"현재 리뷰 날짜: {current_date}, 리뷰수: {len(reviews)} ", end="\r")
    except:
        reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
        print(f"현재 리뷰 날짜: {current_date}, 리뷰수: {len(reviews)} ", end="\r")
    
    if current_date < end_date:
        break

########################################################################
# 리뷰 추출 부분

all_result = []
for review in tqdm(reviews):
    # 리뷰일
    review_date = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
    # 리뷰일을 날짜형 데이터로 변경
    # datetime.strptime(yyyy-mm-dd, "%Y-%m-%d") 날짜형 데이터 타입으로 변환
    review_date = review_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
    review_date = datetime.strptime(review_date, "%Y-%m-%d")
    # 별점
    rating = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > div").get_attribute("aria-label").split()[3][0]
    # 사용자리뷰
    user_review = review.find_element(By.CSS_SELECTOR, "div.h3YV2d").text
    
    try:
        # 업체 댓글
        reply = review.find_element(By.CSS_SELECTOR, "div.ocpBU div.ras4vb > div").get_attribute("innerHTML")
    except:
        reply = None

    result = (review_date, rating, user_review, reply)
    all_result.append(result)
result_df = pd.DataFrame(all_result, columns=['리뷰일', '평점', '사용자리뷰', '업체답변'])
result_df.to_csv("./scraping_results/viva_republica_toss_review_result2.csv", index=False, encoding="utf-8-sig")

100%|████████████████████████████████████████████████████████████████████████████████| 160/160 [00:08<00:00, 19.64it/s]


# 여러 앱 리뷰 수집하기

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime, timedelta
# import datetime
import pandas as pd
import time
from tqdm import tqdm

options = Options()
options.add_experimental_option("detach", True)
options.add_argument("start-maximized")
options.add_argument("Chrome/135.0.0.0")
options.add_argument("lang=ko_KR")

# 웹브라우저가 백그라운드에서 작동하도록 설정
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
    )

bank_app_list = ['viva.republica.toss', 'com.shinhan.sbanking', 
                 'com.kbstar.kbbank', 'com.kebhana.hanapush',
                 'com.wooribank.smart.npib', 'com.rainist.banksalad2']

for bank in bank_app_list:
    url = f"https://play.google.com/store/apps/details?id={bank}"
    driver.get(url)

# 평점 및 리뷰 옆의 -> 버튼 클릭
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label*='평점 및 리뷰 자세히 알아보기']")))
    button.click()

    # 최신순으로 리뷰를 정렬하기 위해서 버튼 클릭
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
    button.click()

    time.sleep(1)

    # 최신을 찾아 클릭
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "span[aria-label*='최신']")))
    button.click()

    time.sleep(3)

    # 오늘로부터 3년전 날짜 만들기
    today = datetime.today()
    end_date = today - timedelta(days=5)
    current_date = ""
    reviews = ""

    while True:

        try:
            # 리뷰가 담긴 창을 찾아서 JavaScript로 1000px씩 아래로 스크롤
            driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 1000)")
            time.sleep(1)

            # 리뷰들 중 가장 마지막 데이터의 날짜 추출해 날짜 데이터로 변환
            reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
            current_date = reviews[-1].find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
            current_date = current_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
            current_date = datetime.strptime(current_date, "%Y-%m-%d")
            print(f"{bank} 현재 리뷰 날짜: {current_date}, 리뷰수: {len(reviews)} ", end="\r")
        except:
            reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
            print(f"현재 리뷰 날짜: {current_date}, 리뷰수: {len(reviews)} ", end="\r")

        if current_date < end_date:
            break

    ########################################################################
    # 리뷰 추출 부분

    all_result = []
    for review in tqdm(reviews):
        # 리뷰일
        review_date = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
        # 리뷰일을 날짜형 데이터로 변경
        # datetime.strptime(yyyy-mm-dd, "%Y-%m-%d") 날짜형 데이터 타입으로 변환
        review_date = review_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        review_date = datetime.strptime(review_date, "%Y-%m-%d")
        # 별점
        rating = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > div").get_attribute("aria-label").split()[3][0]
        # 사용자리뷰
        user_review = review.find_element(By.CSS_SELECTOR, "div.h3YV2d").text

        try:
            # 업체 댓글
            reply = review.find_element(By.CSS_SELECTOR, "div.ocpBU div.ras4vb > div").get_attribute("innerHTML")
        except:
            reply = None

        result = (review_date, rating, user_review, reply)
        all_result.append(result)
    result_df = pd.DataFrame(all_result, columns=['리뷰일', '평점', '사용자리뷰', '업체답변'])
    result_df.to_csv(f"./scraping_results/{bank}_review_result2.csv", index=False, encoding="utf-8-sig")
driver.close()



100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [00:03<00:00, 19.57it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 22.37it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 80/80 [00:03<00:00, 20.38it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 20.19it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 20.76it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 19.88it/s]


# 여러 은행 앱의 리뷰 데이터 수집하기 함수화

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime, timedelta
# import datetime
import pandas as pd
import time
from tqdm import tqdm
from bankdbio import to_bank_db

In [2]:
def bank_app_reviews(bank, year=1):
    options = Options()
    options.add_experimental_option("detach", True)
    options.add_argument("start-maximized")
    options.add_argument("Chrome/135.0.0.0")
    options.add_argument("lang=ko_KR")
    # 웹브라우저가 백그라운드에서 작동하도록 설정
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")


    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
        )

    url = f"https://play.google.com/store/apps/details?id={bank}"
    driver.get(url)

    # 평점 및 리뷰 옆의 -> 버튼 클릭
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label*='평점 및 리뷰 자세히 알아보기']")))
    button.click()

    # 최신순으로 리뷰를 정렬하기 위해서 버튼 클릭
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
    button.click()

    time.sleep(1)

    # 최신을 찾아 클릭
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "span[aria-label*='최신']")))
    button.click()

    time.sleep(3)

    # 오늘로부터 1달 전 날짜 만들기
    today = datetime.today()
    end_date = today - timedelta(days=year*365)
    current_date = ""
    reviews = ""
    pre_n_reviews = ""
    while True:
        try:
            # 스크롤
            driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 1000)")
            time.sleep(1)

            # 리뷰 가져오기
            reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
            current_date = reviews[-1].find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
            current_date = current_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
            current_date = datetime.strptime(current_date, "%Y-%m-%d")

            print(f"{bank} 현재 리뷰 날짜: {current_date}, 리뷰수: {len(reviews)} ", end="\r")
        except:
            reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
            print(f"{bank} 현재 리뷰 날짜: {current_date}, 리뷰수: {len(reviews)} ", end="\r")

        # 종료 조건
        if current_date < end_date:
            print(f"\n {bank}: {year}년 이전 리뷰 도달. 종료.")
            break
            if len(reviews) == pre_n_reviews:
                print(f"\n {bank}: 더 이상 새로운 리뷰 없음. 종료.")
                break

        # 업데이트 위치를 이곳으로 옮김
        pre_n_reviews = len(reviews)
     
    return driver, reviews

In [3]:
def review_extraction(driver, reviews):    
    all_result = []
    for review in tqdm(reviews):
        # 리뷰일
        review_date = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
        # 리뷰일을 날짜형 데이터로 변경
        # datetime.strptime(yyyy-mm-dd, "%Y-%m-%d") 날짜형 데이터 타입으로 변환
        review_date = review_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        review_date = datetime.strptime(review_date, "%Y-%m-%d")
        # 별점
        rating = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > div").get_attribute("aria-label").split()[3][0]
        # 사용자리뷰
        user_review = review.find_element(By.CSS_SELECTOR, "div.h3YV2d").text

        try:
            # 업체 댓글
            reply = review.find_element(By.CSS_SELECTOR, "div.ocpBU div.ras4vb > div").get_attribute("innerHTML")
        except:
            reply = None

        result = (review_date, rating, user_review, reply)
        all_result.append(result)
    result_df = pd.DataFrame(all_result, columns=['리뷰일', '평점', '사용자리뷰', '업체답변'])
#     result_df.to_csv(f"./scraping_results/{bank}_review_result.csv", index=False, encoding="utf-8-sig")
    to_bank_db(bank, result_df)
    print(f"{bank} 리뷰 저장 완료")
    driver.close()  
    


In [ ]:
bank_app_list = ['viva.republica.toss', 'com.shinhan.sbanking', 
                 'com.kbstar.kbbank', 'com.kebhana.hanapush',
                 'com.wooribank.smart.npib', 'com.rainist.banksalad2']

for bank in bank_app_list[0:1]:
    driver, reviews = bank_app_reviews(bank, 3)
    review_extraction(driver, reviews)